## Step 1: Installing Required Libraries for RAG-Based Healthcare Agent

### Objective
To install all necessary dependencies required to build a Retrieval-Augmented Generation (RAG) system using LangChain, ChromaDB, HuggingFace embeddings, and Groq LLM.

### Why This Step is Important
- LangChain enables agentic workflows and tool integration.
- ChromaDB allows persistent vector storage for document retrieval.
- HuggingFace embeddings provide stable, local embedding generation.
- Groq provides high-performance LLM inference.
- Using stable and minimal dependencies avoids runtime conflicts.

This setup ensures a production-ready, stable environment for our healthcare AI agent.


In [ ]:
!pip install -qU \
langchain \
langchain-community \
langchain-chroma \
chromadb \
pypdf \
sentence-transformers \
langchain-groq


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/2

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

print("Imports working fine")


Imports working fine


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Checking the Folder is created for storing the files and data

In [ ]:
import os

project_path = "/content/drive/MyDrive/health_rag_project"
os.makedirs(project_path, exist_ok=True)

print("Folder ready:", project_path)


Folder ready: /content/drive/MyDrive/health_rag_project


## Loading the pdf using the pdf loader in the notebook

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "/content/drive/MyDrive/health_rag_project/AHA-GUIDELINEDRIVEN-MANAGEMENT-OF-HYPERTENSION--AN-EVIDENCEBASED-UPDATE.pdf"
pdf2 = "/content/drive/MyDrive/health_rag_project/guidance-on-global-monitoring-for-diabetes.pdf"

loader1 = PyPDFLoader(pdf1)
loader2 = PyPDFLoader(pdf2)

docs1 = loader1.load()
docs2 = loader2.load()

all_docs = docs1 + docs2

print("Total pages loaded:", len(all_docs))


Total pages loaded: 150


## Creating Chunks of the Data because medical Dataset is heavy

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(all_docs)

print("Total chunks created:", len(chunks))


Total chunks created: 552


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings ready")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings ready


In [ ]:
from langchain_chroma import Chroma

import shutil
import os

db_path = "/content/drive/MyDrive/health_rag_project/vector_db"

if os.path.exists(db_path):
    shutil.rmtree(db_path)
    print("Old vector DB deleted.")
else:
    print("No existing DB found.")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="/content/drive/MyDrive/health_rag_project/vector_db"
)

print("Vector DB ready")
print("Vector count:", vector_store._collection.count())
print("Expected chunks:", len(chunks))

No existing DB found.


InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database

## Step: Loading API Keys Securely from Google Colab Secrets

### Objective
To securely access API keys stored in Google Colab's Secrets Manager without hardcoding them in the notebook.

### Why This Step is Important
- Prevents exposing API keys in code.
- Keeps notebook secure and shareable.
- Ensures proper environment variable setup for LLM access.

We will retrieve the GROQ_API_KEY from Colab Secrets and set it as an environment variable.


In [ ]:
import os
from google.colab import userdata

# Retrieve key from Colab Secrets
groq_key = userdata.get("GROQ_API_KEY")

# Set environment variable
os.environ["GROQ_API_KEY"] = groq_key

print("GROQ key loaded successfully")


In [ ]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Groq LLM initialized")


Groq LLM initialized


## Step: Creating a Retriever from the Vector Database

### Objective
To convert the Chroma vector database into a retriever that can fetch the most relevant document chunks based on a user query.

### Why This Step is Important
- The retriever is the core component of a RAG system.
- It allows the agent to fetch contextually relevant medical information.
- Instead of sending entire documents to the LLM, we send only the most relevant chunks.
- This improves accuracy, reduces hallucination, and increases efficiency.

We will configure the retriever to return the top 3 most relevant chunks for each query.


In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 6
    }
)

## Step: Defining a Retrieval Tool for the Healthcare Agent

### Objective
To create a structured tool that allows the LLM to retrieve relevant medical information from the vector database when required.

### Why This Step is Important
- In an agentic system, the LLM does not directly access the database.
- Instead, it decides when to call a tool.
- The tool fetches relevant medical chunks from the retriever.
- This reduces hallucinations and ensures responses are grounded in official medical documents (WHO & AHA guidelines).

This tool will act as the knowledge access layer of our Healthcare AI Agent.


In [ ]:
from langchain_core.tools import tool

@tool
def get_medical_guidance(query: str) -> str:
    results = retriever.invoke(query)

    # LIMIT each chunk to 500 characters
    combined_text = "\n\n".join(
        [doc.page_content[:500] for doc in results]
    )

    return combined_text
print("Medical retrieval tool ready")


Medical retrieval tool ready


## Step: Binding the Medical Retrieval Tool to the LLM

### Objective
To connect the Groq LLM with the medical retrieval tool so that it can decide when to call the tool during reasoning.

### Why This Step is Important
- This enables tool-calling capability.
- The LLM can dynamically retrieve medical guidance instead of hallucinating answers.
- This transforms a simple chatbot into an agentic system.

We will bind the `get_medical_guidance` tool to the Groq LLM.


In [ ]:
llm_with_tools = llm.bind_tools([get_medical_guidance])

print("LLM successfully bound to medical tool")


LLM successfully bound to medical tool


## Step: Implementing the Agentic Reasoning Loop

### Objective
To create an interactive reasoning loop where the LLM can:
1. Understand the user's question.
2. Decide whether to call a tool.
3. Execute the tool if needed.
4. Use the retrieved information to generate a grounded response.

### Why This Step is Important
- This creates true agentic behavior.
- The LLM can dynamically fetch knowledge from medical guidelines.
- It reduces hallucination by forcing grounded responses.
- It mimics real-world autonomous reasoning systems.

We will implement a manual tool-calling loop similar to the workshop structure.


In [ ]:
# ==============================
# Healthcare RAG Agent - Final Interactive Version
# ==============================

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# ---- System Prompt ----
system_prompt = SystemMessage(
    content="""
You are a professional healthcare AI assistant.

Rules:
1. Answer ONLY using retrieved medical guidelines.
2. Do NOT add extra general advice.
3. Do NOT generate your own disclaimer.
4. If information is missing, say you do not have enough data.
5. Provide structured medical responses.
"""
)

# ---- Agent Function ----
def run_health_agent(user_query):

    messages = [
        system_prompt,
        HumanMessage(content=user_query)
    ]

    while True:
        response = llm_with_tools.invoke(messages)

        # If LLM calls tool
        if response.tool_calls:
            for tool_call in response.tool_calls:

                tool_name = tool_call["name"]
                tool_args = tool_call["args"]

                if tool_name == "get_medical_guidance":

                    tool_result = get_medical_guidance.invoke(
                        tool_args["query"]
                    )

                    messages.append(response)
                    messages.append(
                        ToolMessage(
                            content=tool_result,
                            tool_call_id=tool_call["id"]
                        )
                    )
        else:
            final_answer = response.content

            disclaimer = "\n\n⚠️ Disclaimer: This information is for educational purposes only and is not a substitute for professional medical advice."

            return final_answer + disclaimer


# ---- Interactive Loop ----
print("🏥 Healthcare RAG Agent Ready!")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("👤 You: ")

    if user_input.lower() == "exit":
        print("Agent stopped.")
        break

    response = run_health_agent(user_input)

    print("\n🤖 AI Doctor:\n")
    print(response)
    print("\n" + "="*60 + "\n")

🏥 Healthcare RAG Agent Ready!
Type 'exit' to stop.

👤 You: What is Stage 2 hypertension?


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01k9shcfvwenttkcm314ferw3c` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6485, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}